### Collect zotero entries and their attachments, store the results in a file

**Warning**: misses most of what's in my Zotero DB.  
            *Maybe it's only getting entries I made since switching to zotero 7?*

In [5]:
%load_ext autoreload
%autoreload 2

from icecream import ic
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import refwrangle.utils.refwrangle as rfw
import matplotlib.pyplot as plt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
def get_first_creator(item):
    """Get first creator (usually author) from zotero db top level parent item"""
    # Check if creators key exists and is not empty
    if 'creators' in item['data'] and item['data']['creators']:
        creators = item['data']['creators']
        for creator in creators:
            if creator['creatorType'] == 'author':
                if 'name' in creator:
                    return creator['name']
                else:
                    return f"{creator['lastName']}, {creator['firstName']}"
    return ''  # no creators found

def get_item_venue(item):
    """Get the venue where item appeared.  Zotero puts this in many different fields"""
    data = item['data']
    
    # Check different possible venue fields in order of priority: I know these exist
    venueKeyInPriority = [
        'publicationTitle',  # For journal articles
        'journalAbbreviation', # For journal articles
        'bookTitle',         # For book chapters
        'publisher',         # For books
        'proceedingsTitle',  # For conference papers
        'blogTitle',         # For blog posts
        'websiteTitle',      # For web pages
        'encyclopediaTitle', # For encyclopedia articles
        'dictionaryTitle',   # For dictionary entries
        'conferenceName',    # For conference papers
        'university',        # For theses
        'publisher',         # For books
        'institution',       # For reports
        'libraryCatalog',    # For library catalog entries (how zotero files YouTube)
        'place',             # For location
    ]
    
    for field in venueKeyInPriority:
        if field in data and data[field]:
            return data[field] # assume field is the venue key
        
    # If here, didn't find any of the priority venues.
    # Try to find one in this possibly ficticious dict from perplexity
    # https://www.perplexity.ai/search/for-the-item-type-forum-post-w-_4Myygs7Qni_.0iVwQ3vLg#3

    venueKeyForItemType = {
        'blogPost': 'blogTitle',
        'book': 'publisher',
        'bookSection': 'bookTitle',
        'computerProgram': 'company',
        'conferencePaper': 'proceedingsTitle',
        'dataset': 'repository',
        'dictionaryEntry': 'dictionaryTitle',
        'document': 'archive',
        'email': 'subject',
        'encyclopediaArticle': 'encyclopediaTitle',
        'forumPost': 'forumTitle',
        'journalArticle': 'publicationTitle',
        'magazineArticle': 'publicationTitle',
        'manuscript': 'archive',
        'newspaperArticle': 'publicationTitle',
        'note': 'note',
        'preprint': 'repository',
        'presentation': 'conferenceName',
        'report': 'institution',
        'thesis': 'university',
        'videoRecording': 'libraryCatalog',
        'webpage': 'websiteTitle'
    }

    try:
        itemType = data['itemType']
        return venueKeyForItemType[itemType]
    except:
        print(f"failed to find venue for {rfw.get_citation_key(data)}")
        return ''

def get_parent_metadata(parent_item, collection_names):
    """Parse a parent_item's metadata into a dict w/ standardized names in the keys"""

    # Get "author": can be many things in zotero
    pdat = parent_item['data']
    firstCreator = ''
    if 'creators' in pdat and pdat['creators']:
        creators = pdat['creators']
        ctypes = [creator['creatorType'] for creator in creators]
        hasAuthor = 'author' in ctypes # so can prioritize author creator
        for creator in creators:
            if (creator['creatorType'] == 'author') or not hasAuthor:
                if 'name' in creator:
                    firstCreator = creator['name']
                else:
                    firstCreator = f"{creator['lastName']}, {creator['firstName']}"
                break

    def get_if_there(pkey):
        return pdat[pkey] if pkey in pdat else ''

    return dict(parentFirstCreator = firstCreator,
                # convert collections from zotero keys to names
                parentCollections=[collection_names[colkey] for colkey in pdat['collections']],
                
                parentCitekey=rfw.get_citation_key(pdat),
                parentVenue=get_item_venue(parent_item),
                parentDate = get_if_there('date'),
                parentTitle = get_if_there('title'),
                parentURL=get_if_there('url'),
                parentZotkey=pdat['key'])

In [7]:
# Read and parse the zotero database

# Remote API access
zot = zotero.Zotero(rfw.zotero_library_id, rfw.zotero_library_type, rfw.zotero_api_key)

collection_names = (defaultdict(str) # handle missed collection (W5HNMQSX for parent QTESUD23)
                    | {collection['key']: collection['data']['name'] for collection in zot.collections()})

# Get all pdf and html attachments and associate them with their parent info
# parentItems = zot.everything(zot.top())

zotero_cache = rfw.ZoteroCache()
parentItems = zotero_cache.get_data()

ic| self.filename: WindowsPath('C:/Users/scott/repos/refwrangle/src/refwrangle/utils/dat/orig/proc/ZoteroDBcache.bin')


Reading from cache.


In [8]:
child_exceptions, attachment_files = [], []
def process_parent_items(parentItems, collection_names):
    child_exceptions, attachment_files = [], []
    for parent in parentItems:
        if len(parent['meta']) < 1:
            continue # skip standalone notes or entries e.g. topictags.org

        pdat_always_save = get_parent_metadata(parent, collection_names)

        if rfw.is_youtube_video(parent):
            sourceInfo = pdat_always_save.copy()
            sourceInfo['contentType'] = 'youtube_video'
            attachment_files.append(sourceInfo) # not truly a file: info comes from URL
            continue

        parentCitekey = pdat_always_save['parentCitekey']
        errorParentIDstr = f'[{parentCitekey}]: {pdat_always_save["parentTitle"]}'
        for child in zot.children(parent['key']):
            if rfw.is_ignorable_child(child):
                continue

            cdat = child['data'] | pdat_always_save
            fixed = {'Fixed':False}
            try:
                cdat['file_basename'] = cdat['path'].removeprefix("attachments:")
                guessFNm = rfw.lit_attachment_dir_shared / cdat['file_basename']
                if guessFNm.exists():
                    cdat['file_fullpath'] = guessFNm
                else:
                    errStr = f'Error for {errorParentIDstr}. Full path does not exist: "{guessFNm}"'
                    print(errStr)
                    child_exceptions.append({'exception':errStr} | cdat | fixed)
            except Exception as e:
                print(f'Error for  {errorParentIDstr}: {e}')
                # No idea why these errors occur.  Try to fix
                for ext in ['pdf', 'html']: 
                    guessBasename = f'{parentCitekey}.{ext}'
                    guessFNm = rfw.lit_attachment_dir_shared / guessBasename
                    if guessFNm.exists():
                        cdat['file_basename'] = guessBasename
                        cdat['file_fullpath'] = guessFNm
                        break # if find pdf first, don't get html
                if  'file_basename' in cdat:
                    print(f"\tFix by guess basename worked: {cdat['file_basename']}")
                    fixed = {'Fixed':True}
                else:
                    print(f'\tCould not fix it. Full path does not exist: "{guessFNm}"')
                    fixed = {'Fixed':False}
                    continue # no html or pdf: don't allow it in attachment_files (below)

                child_exceptions.append({'exception': str(e)} | cdat | fixed)

            attachment_files.append(cdat)

    attachment_files = pd.DataFrame(attachment_files)
    child_exceptions = pd.DataFrame(child_exceptions)
    return child_exceptions, attachment_files

child_exceptions, attachment_files = process_parent_items(parentItems, collection_names)

attachment_files = pd.DataFrame(attachment_files)
child_exceptions = pd.DataFrame(child_exceptions)

Error for [FAIR24fairVsWalkerLoanDiscrim]: FAIR v. Walker. Full path does not exist: "C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\lit\lit_sources\FAIR24fairVsWalkerLoanDiscrim.pdf"
Error for [Grose24youngWimynWillTalkSexistm]: Young Women Will Never Stop Talking About Sexism. Full path does not exist: "C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\lit\lit_sources\Grose24youngWimynWillTalkSexistm.html"
Error for [Abdel-Nasser19pvPowFrcst-LSTM-RNN]: Accurate photovoltaic power forecasting models using deep LSTM-RNN. Full path does not exist: "C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\lit\lit_sources\Abdel-Nasser19pvPowFrcst-LSTM-RNN.pdf"
Error for [Smith10ModelingLongitudinalData]: Modeling Longitudinal Data Using a Pair-Copula Decomposition of Serial Dependence. Full path does not exist: "C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\lit\lit_sources\Smith10ModelingLongitudinalData.pdf"
Error for [Dionne24hidde

In [9]:
collection_key_to_name = {collection['key']:collection['data']['name'] for collection in zot.all_collections()} 
ic(collection_key_to_name)


ic| collection_key_to_name: {'2JJ77KLR': 'peakFrcst',
                             '2U6RDTCP': 'copula',
                             '2UQTBFN2': 'Optimization',
                             '2W6YI2EF': 'Forecast General',
                             '376WMMQJ': 'Multitask Learning',
                             '38B5XBNU': 'AI, General',
                             '3A79UTTN': 'Mutual Information',
                             '3H82PKV7': 'Calibration of Classifiers',
                             '3IHPD3RF': 'Data Sources, Political',
                             '3PFYX9N9': 'priceNormalization',
                             '3RF59QPA': 'Load vs. Traffic',
                             '3VJREPFF': 'Strength Training',
                             '3ZBI89RM': 'priceFrcstAEMO',
                             '4E5CFYAQ': 'Forecast Wind',
                             '4HRM8T3W': 'featSel',
                             '4JXTCBPP': 'distribOpt',
                             '4TT83CPJ': 'Sola

{'NFHWS5LI': 'thesis',
 'QA3AUC44': 'article1',
 'RW86YUP8': 'Scratch Space',
 '98ADG6FD': 'Basic Stats',
 'FC2AYPI7': 'Dependency Metrics (merge somewhere)',
 'BJU7YNSE': 'Obsidan Note to Make',
 '9BC2FPZT': 'Data Bases',
 'XF53YK6P': 'Electric Design',
 '7XE2QVM9': 'Lighting',
 'ETXAUSS7': 'Electric Motors',
 'UV7SZ6I6': 'Politics',
 'EALY7KTD': 'Short Termism',
 'MZZ9DF2D': 'Non-Electoral Politics',
 'RWCZBYN7': 'Economic Politics',
 'X9K7P8XU': 'Morality in Politics',
 'F5BSZNTZ': 'Authoritarianism',
 '3IHPD3RF': 'Data Sources, Political',
 'JF8WPLJ7': 'SuccessStories',
 'RTW4J2UL': 'BizBrandTools',
 'II7T84JD': 'MisDisinformation',
 'Z9U4FYTX': 'PartyOrganization',
 'CBAIHAHS': 'Polarization',
 'KMG9TMGD': 'PoliticalML',
 'MZCMVMZK': 'FocusGroups',
 'FWUA2JE9': 'IdentityPolitics',
 'A49HZQVA': 'MediaAdsPolit',
 'IXYGRBEW': 'ElectionPredFeats',
 'HFHS63BN': 'NeuroPsychoLinguisticPolitics',
 'URZKREHG': 'Hot Takes US Elect 2024',
 'NM45ABSL': 'CampaignMoney',
 'VCILTFIN': 'CivicSatC

In [10]:
unfixed = child_exceptions.query('Fixed != True')
if (nUnfixed := len(unfixed)) > 0:
    print(f'{nUnfixed} unfixed child exceptions')
    display(unfixed)
else:
    print(f'Fixed {child_exceptions.Fixed.value_counts().values[0]} of {len(child_exceptions)} exceptions')

71 unfixed child exceptions


,exception,key,version,parentItem,itemType,linkMode,title,accessDate,url,note,...,parentDate,parentTitle,parentURL,parentZotkey,file_basename,Fixed,filename,md5,mtime,file_fullpath
0,Error for [FAIR24fairVsWalkerLoanDiscrim]: FAI...,ZFQAIIPK,21565,KWGDTCTL,attachment,linked_file,FAIR24fairVsWalkerLoanDiscrim.pdf,,,,...,"October 29, 2024",FAIR v. Walker,https://www.fairforall.org/fair-v-walker/,KWGDTCTL,FAIR24fairVsWalkerLoanDiscrim.pdf,False,NaN,NaN,NaN,NaN
1,Error for [Grose24youngWimynWillTalkSexistm]: ...,5RAHJBS7,21638,HLRPQ4NP,attachment,linked_file,Grose24youngWimynWillTalkSexistm.html,2025-01-19T16:13:36Z,https://www.nytimes.com/2024/11/20/opinion/tru...,,...,2024-11-20,Young Women Will Never Stop Talking About Sexism,https://www.nytimes.com/2024/11/20/opinion/tru...,HLRPQ4NP,Grose24youngWimynWillTalkSexistm.html,False,NaN,NaN,NaN,NaN
2,Error for [Abdel-Nasser19pvPowFrcst-LSTM-RNN]:...,GS9WBY9Y,4465,8HIUG7L8,attachment,linked_file,Abdel-Nasser19pvPowFrcst-LSTM-RNN.pdf,,,"<p xmlns=""http://www.w3.org/1999/xhtml"" id=""ti...",...,2019-07-01,Accurate photovoltaic power forecasting models...,https://doi.org/10.1007/s00521-017-3225-z,8HIUG7L8,Abdel-Nasser19pvPowFrcst-LSTM-RNN.pdf,False,NaN,NaN,NaN,NaN
3,Error for [Smith10ModelingLongitudinalData]: M...,B5MVREAL,22256,97G685E5,attachment,linked_file,Smith10ModelingLongitudinalData.pdf,,,,...,2010-12-01,Modeling Longitudinal Data Using a Pair-Copula...,https://www.tandfonline.com/doi/full/10.1198/j...,97G685E5,Smith10ModelingLongitudinalData.pdf,False,NaN,NaN,NaN,NaN
4,Error for [Dionne24hiddenVictoryProgrssiv]: Th...,A5RX93Y7,22224,4B389XPQ,attachment,linked_file,Dionne24hiddenVictoryProgrssiv.html,,,,...,2024-11-24,The lessons in progressives’ hidden 2024 victo...,https://www.washingtonpost.com/opinions/2024/1...,4B389XPQ,Dionne24hiddenVictoryProgrssiv.html,False,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,Error for [Bidgely19amInsightsRprt]: AMI-Drive...,JIPHE93A,14479,EKLZCYP5,attachment,linked_file,Bidgely19amInsightsRprt.pdf,,,,...,August 2019,AMI-Driven Insights Report,https://www.idcutilitiessummit.com/index/RESOU...,EKLZCYP5,C:\Users\scott\OneDrive\share\ref\zotero\paper...,False,NaN,NaN,NaN,NaN
78,Error for [Mayhorn16disaggLdRealWrldPerf]: Loa...,62I2GBJK,14480,RCMMDNV6,attachment,linked_file,Mayhorn16disaggLdRealWrldPerf.pdf,,,,...,2016,Load Disaggregation Technologies: Real World a...,,RCMMDNV6,C:\Users\scott\OneDrive\share\ref\zotero\paper...,False,NaN,NaN,NaN,NaN
79,Error for [Hare18disaggHmLdDmdResp]: Disaggreg...,SX6AFLGY,14480,WA8IQAXP,attachment,linked_file,Hare18disaggHmLdDmdResp.pdf,,,,...,2018,Disaggregation of residential home energy via ...,https://dspace.mit.edu/handle/1721.1/117983,WA8IQAXP,C:\Users\scott\OneDrive\share\ref\zotero\paper...,False,NaN,NaN,NaN,NaN
80,Error for [Rehman21LoadDisaggThesis]: Load Dis...,9G5G5RPX,14481,8MAAZN8P,attachment,linked_file,Rehman21LoadDisaggThesis.pdf,,,,...,2021,Load Disaggregation: Towards Energy Efficient ...,https://openrepository.aut.ac.nz/handle/10292/...,8MAAZN8P,C:\Users\scott\OneDrive\share\ref\zotero\paper...,False,NaN,NaN,NaN,NaN


In [11]:
# Find entries with unclassified venu

unclassifiedVenues = []
for parent in parentItems:
    pinfo = get_parent_metadata(parent, collection_names)
    if len(pinfo['parentVenue'])<1:
        unclassifiedVenues.append(pinfo)

if (nUnclassifVenues := len(unclassifiedVenues)) > 0:
    print(f'There were {nUnclassifVenues} unclassified venues:')
    unclassifiedVenues = pd.DataFrame(unclassifiedVenues)
    display(unclassifiedVenues)
else:
    print('No unclassified venues')

No unclassified venues


In [12]:
# Count number of attachments (not parents) per collection
citekeysInCollection = defaultdict(list)
for row in attachment_files.itertuples(index=False):
    for collection in row.parentCollections:
        try:
            citekeysInCollection[collection].append(row.parentCitekey)
        except Exception as e:
            ic(e, row)

collection_counts = pd.Series({collection: len(filekeys) for collection, filekeys in citekeysInCollection.items()})
collection_counts.sort_values(ascending=False, inplace=True)
# plt.figure(figsize=(5, 10))  # Width is set to 8 inches, height to 5 inches
# collection_counts.plot(kind='barh',)

In [13]:
rfw.save_pickle_data(rfw.extractedZoteroEntriesFNm, 
                 {'attachment_files':attachment_files, 
                 'child_exceptions':child_exceptions,
                 'collection_counts':collection_counts})

Writing to C:\Users\scott\repos\refwrangle\src\refwrangle\utils\dat\zotero_entries.pkl...


In [14]:
#child_exceptions.query('Fixed != True')
unfixed.loc[1].exception

'Error for [Grose24youngWimynWillTalkSexistm]: Young Women Will Never Stop Talking About Sexism. Full path does not exist: "C:\\Users\\scott\\OneDrive\\share\\ref\\obsidian\\Obsidian Share Vault\\lit\\lit_sources\\Grose24youngWimynWillTalkSexistm.html"'

In [15]:
collection_counts[collection_counts> 6]

                                        440
Generative AI                           313
Hot Takes US Elect 2024                 307
NeuroPsychoLinguisticPolitics           148
MediaAdsPolit                           126
PoliticalML                              71
IdentityPolitics                         64
priceFrcstAEMO                           56
Market Design Electricity                46
PollMethods                              45
Conformal Prediction                     45
Polarization                             43
ElectionPredFeats                        40
battFires                                37
Voting Systems                           36
MisDisinformation                        34
Forecast aggr_disaggr                    29
CAISO Market                             25
FocusGroups                              21
Contextual Optimization                  18
priceSpikeVolatile                       17
Scratch Space                            17
Optimization                    